# Phase 1 — GRPO-train an LLM portfolio manager (pipeline validation)

**Goal of this run:** prove the whole pipeline works end-to-end (data → prompts → GRPO → backtest)
and get a first honest baseline. We deliberately use a small model (Qwen3-4B) to keep this cheap
(~15–20 compute units); Phase 2/3 scale up to a 14B coding agent.

**The loop** (see `docs/rl_trading_explained.html` §6 for the full theory):
1. A historical market day is rendered as a text prompt (12 ETF features).
2. The model samples **8 different allocations** for that same day.
3. Each is scored by the *realized* return over the following 5 trading days.
4. GRPO normalizes rewards *within the group of 8*, so "the market went up" cancels out
   and only relative allocation skill is rewarded.

**Setup:** Runtime → Change runtime type → **A100 GPU**.

**What we expect to see:** invalid-output rate → 0 within ~50 steps, then a *gentle* upward
drift in mean reward. No hockey sticks — weekly market returns are ~92% noise (doc §13).

In [ ]:
%%capture
# Unsloth = fast LoRA training kernels + an in-process vLLM engine, so GRPO's
# "generate 8 completions per prompt" step doesn't need a second copy of the model.
!pip install unsloth vllm
!pip install yfinance pyarrow

## 1. Get the project code and data

The repo ships the training data as JSONL (`data/train.jsonl` etc.), so Colab never needs
to talk to Yahoo Finance. Each line is one market day:
```json
{"date": "2010-04-08",
 "prompt": "You are a systematic portfolio manager...",   // what the model sees
 "fwd_returns": {"SPY": 0.0212, ...}}                     // hidden future, used ONLY for scoring
```
The `fwd_returns` are never shown to the model — they are the answer key the reward
function grades against *after* the model has committed to an allocation.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/rl-finance.git"  # <-- set me

import os, sys
if not os.path.exists("rl-finance"):
    !git clone {REPO_URL} rl-finance
sys.path.insert(0, "rl-finance/src")

from pathlib import Path
from rl_finance.data import read_jsonl

DATA = Path("rl-finance/data")
train_samples = read_jsonl(DATA / "train.jsonl")   # 2008-2019, 591 weekly decisions
val_samples = read_jsonl(DATA / "val.jsonl")       # 2020-2022, only for evaluation

print(f"{len(train_samples)} train prompts | {len(val_samples)} val prompts\n")
print(train_samples[100]["prompt"])  # look at one actual state, always

## 2. Load the model with a LoRA adapter

We freeze all 4B base weights and train only low-rank adapters (rank 32 ≈ 0.5% of
parameters). Three reasons (doc §7):
- **Memory** — optimizer states exist only for the adapters.
- **Free reference model** — GRPO's KL leash compares against "adapters switched off";
  no second model copy.
- **Regularization** — a rank-32 subspace physically cannot memorize 591 training days.

`fast_inference=True` starts a vLLM engine that shares these weights for fast sampling.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,          # prompt (~450 tok) + reasoning + JSON fits easily
    load_in_4bit=False,           # bf16; we have 80 GB
    fast_inference=True,          # in-process vLLM for GRPO sampling
    max_lora_rank=32,
    gpu_memory_utilization=0.7,   # vLLM's share; leaves room for training tensors
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Build the training dataset

TRL's `GRPOTrainer` wants a `prompt` column (chat format). Any *extra* columns
(`fwd_returns`, `date`) are passed through to our reward function as keyword
arguments — that's how each completion gets scored against its own day's answer key.

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": s["prompt"]}],
        "fwd_returns": s["fwd_returns"],
        "date": s["date"],
    }
    for s in train_samples
]).shuffle(seed=3407)

train_ds

## 4. The evaluation harness (defined before training, used twice)

One function that turns "a model checkpoint" into "a row in the benchmark table":
1. Generate one allocation per validation day at **temperature 0.2** (deployment wants
   the policy's mode; the 1.0 used in training is for exploration).
2. Parse into weights (unparseable → sit in cash — a real system needs *some* default).
3. Run the **exact same backtester** the classic benchmarks go through: weekly
   rebalancing, weight drift, 10 bps transaction costs. Nobody gets a friendlier simulator.

We run it **once before training** (zero-shot baseline: what does the untrained model
already know?) and once after. The difference is what GRPO actually bought us.

In [ ]:
import pandas as pd
from vllm import SamplingParams

from rl_finance.backtest import run_backtest
from rl_finance.benchmarks import BENCHMARKS
from rl_finance.data import SPLITS, download_prices
from rl_finance.metrics import format_table
from rl_finance.rewards import parse_weights

prices = download_prices(cache=DATA / "prices.parquet")


def evaluate_on_val(lora_request=None, label="model"):
    """Backtest a checkpoint on the validation years (2020-2022).

    lora_request=None evaluates the plain base model (zero-shot baseline);
    pass model.load_lora(...) to evaluate a trained adapter.
    """
    chats = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": s["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        for s in val_samples
    ]
    outputs = model.fast_generate(
        chats,
        sampling_params=SamplingParams(temperature=0.2, max_tokens=256),
        lora_request=lora_request,
    )

    decisions, n_invalid = {}, 0
    for s, out in zip(val_samples, outputs):
        w = parse_weights(out.outputs[0].text)
        if w is None:
            n_invalid += 1
            w = {"CASH": 1.0}
        decisions[s["date"]] = w
    print(f"[{label}] {n_invalid}/{len(val_samples)} unparseable outputs")

    policy = lambda date, feats: decisions.get(str(date.date()), {"CASH": 1.0})
    start, end = SPLITS["val"]
    return run_backtest(policy, prices, start, end)["metrics"]


# --- Zero-shot baseline: the untrained model's allocations ---
results = {"llm_zero_shot": evaluate_on_val(label="zero-shot")}
print(format_table(results))

## 5. GRPO training

The knobs, and why (full table in doc §14):

| knob | value | why |
|---|---|---|
| `num_generations` | 8 | group size = quality of the "average sibling" baseline |
| `temperature` | 1.0 | group *diversity* is the learning signal — 8 identical answers teach nothing |
| `learning_rate` | 5e-6 | noisy rewards → small careful steps |
| `num_train_epochs` | 2 | 591 prompts is small; more epochs just re-mines the same noise |
| batch 8 × accum 2 | 16 prompts/step | × 8 generations = 128 scored portfolios per optimizer step |

The reward function is imported from the repo (`rl_finance.rewards.grpo_reward_func`) —
the same code this project's tests and teaching doc use. It parses each completion's JSON
and returns `20 · log(1 + realized_5d_return)`, or **−5** for unparseable output
(why −5 and not −1 is a war story: doc §6, "a real bug this example caught").

**Watch while it trains:** `reward` should drift up gently; `rewards/grpo_reward_func/std`
must stay > 0 (zero = policy collapse); `completions/mean_length` should stay sane (~100-250).

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from rl_finance.rewards import grpo_reward_func

config = GRPOConfig(
    output_dir="outputs",
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_generations=8,
    max_prompt_length=1280,
    max_completion_length=256,
    temperature=1.0,
    num_train_epochs=2,
    logging_steps=5,
    save_steps=50,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[grpo_reward_func],
    args=config,
    train_dataset=train_ds,
)
trainer.train()

## 6. Did it learn anything? Trained model vs. zero-shot vs. classic benchmarks

All rows come from the same backtester on the same 2020–2022 window. Reading guide:
- `llm_grpo` vs `llm_zero_shot` — what RL added beyond the base model's priors.
- `llm_grpo` vs `spy`/`60_40`/`equal_weight` — whether it clears the naive bars.
- Remember the error bars (doc §13): a Sharpe gap under ~0.5 on 3 years is weak evidence.
  Val is for iterating; the untouched test split renders the final verdict, once.

In [ ]:
model.save_lora("grpo_trader_lora")

results["llm_grpo"] = evaluate_on_val(
    lora_request=model.load_lora("grpo_trader_lora"), label="trained"
)

start, end = SPLITS["val"]
for name, policy in BENCHMARKS.items():
    results[name] = run_backtest(policy, prices, start, end)["metrics"]

print(format_table(results))

## 7. Persist the adapter

Colab VMs are ephemeral — copy the LoRA (a few hundred MB) to Drive so the run
outlives the session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/rl-finance && cp -r grpo_trader_lora /content/drive/MyDrive/rl-finance/
print("saved to Drive: rl-finance/grpo_trader_lora")